In [5]:
spark.sql("DROP TABLE IF EXISTS dim_locations")
spark.sql("DROP TABLE IF EXISTS fact_weather_hourly")

StatementMeta(, 4d732621-5ccd-4849-8f3d-4ea126db3ed8, 3, Finished, Available, Finished, False)

DataFrame[]

In [6]:
import uuid
import time
from datetime import date, timedelta
import requests
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from delta.tables import DeltaTable

run_id = str(uuid.uuid4())
failed_cities = []
fact_rows = 0

world_capitals = [
    # Europa
    {"city": "Roma", "lat": 41.9028, "lon": 12.4964, "country": "Italia", "continent": "Europa"},
    {"city": "Parigi", "lat": 48.8566, "lon": 2.3522, "country": "Francia", "continent": "Europa"},
    {"city": "Berlino", "lat": 52.5200, "lon": 13.4050, "country": "Germania", "continent": "Europa"},
    {"city": "Londra", "lat": 51.5074, "lon": -0.1278, "country": "Regno Unito", "continent": "Europa"},
    {"city": "Madrid", "lat": 40.4168, "lon": -3.7038, "country": "Spagna", "continent": "Europa"},
    {"city": "Lisbona", "lat": 38.7223, "lon": -9.1393, "country": "Portogallo", "continent": "Europa"},
    {"city": "Vienna", "lat": 48.2082, "lon": 16.3738, "country": "Austria", "continent": "Europa"},
    {"city": "Atene", "lat": 37.9838, "lon": 23.7275, "country": "Grecia", "continent": "Europa"},
    {"city": "Amsterdam", "lat": 52.3676, "lon": 4.9041, "country": "Paesi Bassi", "continent": "Europa"},
    {"city": "Bruxelles", "lat": 50.8503, "lon": 4.3517, "country": "Belgio", "continent": "Europa"},
    {"city": "Varsavia", "lat": 52.2297, "lon": 21.0122, "country": "Polonia", "continent": "Europa"},

    # Nord America
    {"city": "Washington", "lat": 38.9072, "lon": -77.0369, "country": "Stati Uniti", "continent": "Nord America"},
    {"city": "Ottawa", "lat": 45.4215, "lon": -75.6972, "country": "Canada", "continent": "Nord America"},
    {"city": "Città del Messico", "lat": 19.4326, "lon": -99.1332, "country": "Messico", "continent": "Nord America"},

    # Sud America
    {"city": "Brasilia", "lat": -15.8267, "lon": -47.9218, "country": "Brasile", "continent": "Sud America"},
    {"city": "Buenos Aires", "lat": -34.6037, "lon": -58.3816, "country": "Argentina", "continent": "Sud America"},
    {"city": "Santiago", "lat": -33.4489, "lon": -70.6693, "country": "Cile", "continent": "Sud America"},
    {"city": "Bogotà", "lat": 4.7110, "lon": -74.0721, "country": "Colombia", "continent": "Sud America"},

    # Asia
    {"city": "Tokyo", "lat": 35.6762, "lon": 139.6503, "country": "Giappone", "continent": "Asia"},
    {"city": "Pechino", "lat": 39.9042, "lon": 116.4074, "country": "Cina", "continent": "Asia"},
    {"city": "Nuova Delhi", "lat": 28.6139, "lon": 77.2090, "country": "India", "continent": "Asia"},
    {"city": "Seul", "lat": 37.5665, "lon": 126.9780, "country": "Corea del Sud", "continent": "Asia"},
    {"city": "Bangkok", "lat": 13.7563, "lon": 100.5018, "country": "Thailandia", "continent": "Asia"},

    # Africa
    {"city": "Il Cairo", "lat": 30.0444, "lon": 31.2357, "country": "Egitto", "continent": "Africa"},
    {"city": "Pretoria", "lat": -25.7479, "lon": 28.2293, "country": "Sudafrica", "continent": "Africa"},
    {"city": "Nairobi", "lat": -1.2921, "lon": 36.8219, "country": "Kenya", "continent": "Africa"},
    {"city": "Rabat", "lat": 34.0209, "lon": -6.8416, "country": "Marocco", "continent": "Africa"},

    # Oceania
    {"city": "Canberra", "lat": -35.2809, "lon": 149.1300, "country": "Australia", "continent": "Oceania"},
    {"city": "Wellington", "lat": -41.2865, "lon": 174.7762, "country": "Nuova Zelanda", "continent": "Oceania"}
]

raw_schema = StructType([
    StructField("time", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("city", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("country", StringType(), True),
    StructField("continent", StringType(), True),
    StructField("timezone", StringType(), True),
])

def merge_delta(spark_session, df_source, table_name, keys):
    if not spark_session.catalog.tableExists(table_name):
        df_source.write.format("delta").mode("overwrite").saveAsTable(table_name)
        return

    target = DeltaTable.forName(spark_session, table_name)
    cond = " AND ".join([f"target.{k} = source.{k}" for k in keys])
    updates = {c: f"source.{c}" for c in df_source.columns if c not in keys}

    stmt = target.alias("target").merge(df_source.alias("source"), cond)
    if updates:
        stmt = stmt.whenMatchedUpdate(set=updates)
    stmt.whenNotMatchedInsertAll().execute()

def log_run(status, error=None):
    log_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("status", StringType(), True),
        StructField("fact_rows", StringType(), True),
        StructField("failed_cities", StringType(), True),
        StructField("error", StringType(), True),
    ])
    df_log = (
        spark.createDataFrame([(
            run_id,
            status,
            str(fact_rows),
            ",".join(failed_cities) if failed_cities else None,
            str(error) if error is not None else None
        )], schema=log_schema)
        .withColumn("fact_rows", F.col("fact_rows").cast("int"))
        .withColumn("logged_at", F.current_timestamp())
    )
    df_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("etl_run_log")

try:
    end_dt = date.today().isoformat()
    start_dt = (date.today() - timedelta(days=365)).isoformat()

    all_data = []
    for item in world_capitals:
        url = (
            f"https://archive-api.open-meteo.com/v1/archive"
            f"?latitude={item['lat']}&longitude={item['lon']}"
            f"&start_date={start_dt}&end_date={end_dt}"
            f"&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=auto"
        )
        ok = False
        for attempt in range(3):
            try:
                r = requests.get(url, timeout=60)
                if r.status_code == 200:
                    payload = r.json()
                    hourly = payload["hourly"]
                    df_city = pd.DataFrame({
                        "time": hourly["time"],
                        "temperature": hourly["temperature_2m"],
                        "humidity": hourly["relative_humidity_2m"],
                        "wind_speed": hourly["wind_speed_10m"]
                    })
                    df_city["city"] = item["city"]
                    df_city["latitude"] = payload["latitude"]
                    df_city["longitude"] = payload["longitude"]
                    df_city["country"] = item["country"]
                    df_city["continent"] = item["continent"]
                    df_city["timezone"] = payload.get("timezone", "UTC")
                    all_data.append(df_city)
                    ok = True
                    break
            except Exception:
                time.sleep(2)
        if not ok:
            failed_cities.append(item["city"])
            time.sleep(2)
        time.sleep(0.1)

    assert len(all_data) > 0, "Nessun dato scaricato dall'API."

    df_pandas_total = pd.concat(all_data, ignore_index=True)
    for col in ["time", "city", "country", "continent", "timezone"]:
        df_pandas_total[col] = df_pandas_total[col].astype(str)
    for col in ["temperature", "humidity", "wind_speed", "latitude", "longitude"]:
        df_pandas_total[col] = pd.to_numeric(df_pandas_total[col], errors="coerce")

    df_raw = (
        spark.createDataFrame(df_pandas_total, schema=raw_schema)
        .withColumn("timestamp", F.to_timestamp("time", "yyyy-MM-dd'T'HH:mm"))
        .withColumn("timestamp_id", F.date_format("timestamp", "yyyyMMddHH").cast("int"))
    )

    dim_continents = (
        df_raw.select("continent").distinct()
        .withColumn("continent_id", F.md5("continent"))
    )
    merge_delta(spark, dim_continents, "dim_continents", ["continent_id"])

    dim_countries = (
        df_raw.select("country", "continent").distinct()
        .join(dim_continents, on="continent", how="inner")
        .select("country", "continent_id")
        .withColumn("country_id", F.md5(F.concat_ws("_", "country", "continent_id")))
    )
    merge_delta(spark, dim_countries, "dim_countries", ["country_id"])

    dim_city = (
        df_raw.select("city", "country", "latitude", "longitude", "timezone").distinct()
        .join(dim_countries, on="country", how="inner")
        .withColumn("city_id", F.md5(F.concat_ws("_", "city", "latitude", "longitude")))
        .select("city_id", "city", "country_id", "latitude", "longitude", "timezone")
    )
    merge_delta(spark, dim_city, "dim_city", ["city_id"])

    dim_time = (
        df_raw.select("timestamp", "timestamp_id").distinct()
        .withColumn("date", F.to_date("timestamp"))
        .withColumn("year", F.year("timestamp"))
        .withColumn("month", F.month("timestamp"))
        .withColumn("day", F.dayofmonth("timestamp"))
        .withColumn("quarter", F.quarter("timestamp"))
        .withColumn("hour", F.hour("timestamp"))
        .withColumn("day_of_week", F.date_format("timestamp", "E"))
        .select("timestamp_id", "date", "year", "month", "day", "quarter", "hour", "day_of_week")
    )
    merge_delta(spark, dim_time, "dim_time", ["timestamp_id"])

    df_fact = (
        df_raw
        .join(dim_city.select("city", "latitude", "longitude", "city_id"), on=["city", "latitude", "longitude"], how="inner")
        .select("city_id", "timestamp_id", "temperature", "humidity", "wind_speed")
        .dropDuplicates(["city_id", "timestamp_id"])
    )
    fact_rows = df_fact.count()
    merge_delta(spark, df_fact, "fact_weather_hourly", ["city_id", "timestamp_id"])

    log_run("SUCCESS")
    print(f"Run {run_id} completata. Righe fact nel batch: {fact_rows}")

except Exception as ex:
    log_run("FAILED", str(ex))
    raise

StatementMeta(, 4d732621-5ccd-4849-8f3d-4ea126db3ed8, 4, Finished, Available, Finished, False)

Run 06fb7024-78ca-4993-9268-cd5edbe5fe69 completata. Righe fact nel batch: 254736


In [ ]:
spark.table("etl_run_log").orderBy(F.col("logged_at").desc()).show(5, truncate=False)

for tbl in ["dim_continents", "dim_countries", "dim_city", "dim_time", "fact_weather_hourly"]:
    print(f"{tbl}: {spark.table(tbl).count()} righe")

StatementMeta(, 4d732621-5ccd-4849-8f3d-4ea126db3ed8, 5, Finished, Available, Finished, False)

+------------------------------------+-------+---------+-------------+-----+--------------------------+
|run_id                              |status |fact_rows|failed_cities|error|logged_at                 |
+------------------------------------+-------+---------+-------------+-----+--------------------------+
|06fb7024-78ca-4993-9268-cd5edbe5fe69|SUCCESS|254736   |NULL         |NULL |2026-11-24 13:35:31.869946|
|748356e9-62aa-4d13-9f6f-09e44c0a9f7a|SUCCESS|254736   |NULL         |NULL |2026-11-24 12:59:16.08727 |
|693c061d-98ca-4410-a947-6bef3b2f5473|SUCCESS|4872     |NULL         |NULL |2026-11-19 15:05:26.32558 |
+------------------------------------+-------+---------+-------------+-----+--------------------------+



dim_continents: 6 righe


dim_countries: 29 righe


dim_city: 46 righe


dim_time: 8832 righe
fact_weather_hourly: 254736 righe


In [ ]:
spark.sql("DROP TABLE IF EXISTS weather_forecast_rome")

StatementMeta(, 4d732621-5ccd-4849-8f3d-4ea126db3ed8, 6, Finished, Available, Finished, False)

DataFrame[]